# Exercice : appliquer un extrait « update » KBO sur bronze et silver

Vous disposez d'un dossier d'extrait journalier, par exemple
`KboOpenData_0432_2026_07_26_Update/`, qui contient :

- `meta.csv` : numéro d'extrait (`ExtractNumber`), type (`ExtractType`),
  date du snapshot (`SnapshotDate`)
- pour chaque entité (`enterprise`, `establishment`, `branch`,
  `denomination`, `address`, `contact`, `activity`) : un fichier
  `{entite}_insert.csv` et un fichier `{entite}_delete.csv`

**Important** : pour `enterprise`/`establishment`/`branch`, chaque ligne d'insert
porte une clé unique (`EnterpriseNumber`/`EstablishmentNumber`/`Id`).

Pour `denomination`/`address`/`contact`/`activity`, le fichier delete ne liste
QUE `EntityNumber` (sans clé de ligne précise). Cela signifie que dès qu'UN
SEUL élément change pour une entité (ex : une seule activité NACE ajoutée),
TOUTES les lignes de cette entité dans la table sont supprimées puis réinsérées
en intégralité.

Vous allez :
1. Lire `meta.csv` et les fichiers insert/delete (sans appliquer la même mise à jour deux fois)
2. Appliquer la mise à jour sur bronze
3. Ne reconstruire `entreprise`/`entreprise_silver` QUE pour les entreprises
   réellement touchées par ce lot (pas un rebuild complet)
4. Éviter de rejouer deux fois le même extrait (meta.csv contient un numéro de snapshot)

---

## 0. Ce que l'on construit

La base contient l'extrait **complet n° 431** (snapshot du 24-07-2026), chargé
et transformé par les deux notebooks précédents. Trois extraits journaliers
doivent maintenant être appliqués **dans l'ordre** :

| Extrait | Snapshot | Effet |
|---|---|---|
| 432 | 25-07-2026 | premier delta après le full |
| 433 | 26-07-2026 | suit 432 |
| 434 | 27-07-2026 | suit 433 |

### Deux régimes de mise à jour distincts

Le point que l'énoncé souligne est structurant. Les 7 entités ne se mettent pas
à jour de la même façon :

| Régime | Entités | Fichier delete | Application |
|---|---|---|---|
| **Par clé de ligne** | `enterprise`, `establishment`, `branch` | contient la clé exacte | on supprime cette ligne, on insère les nouvelles |
| **Par entité entière** | `denomination`, `address`, `contact`, `activity` | contient **seulement** `EntityNumber` | on purge **toutes** les lignes de cette entité, puis on réinsère le jeu complet |

Le second régime peut sembler brutal, mais il est logique : ces tables n'ont pas
de clé de ligne stable. Si une entreprise ajoute une activité NACE, la KBO ne
peut pas indiquer « ajoute cette ligne-là » — elle dit « oublie tout ce que tu
sais des activités de cette entité, voici le jeu complet ».

**Conséquence pratique, et c'est ce qui rend le rejeu sûr** : purger puis
réinsérer est une opération **idempotente**. La rejouer deux fois produit le
même état qu'une seule fois.

### L'ordre des opérations est une contrainte, pas un détail

L'énoncé le précise : il faut calculer l'ensemble des entreprises affectées
**avant** d'appliquer les suppressions.

```
        ┌─ 1. calculer l'ensemble AFFECTÉ ────┐   lit le bronze AVANT modification
        │                                      │
        ├─ 2. appliquer deletes + inserts ────┤   le bronze change
        │                                      │
        ├─ 3. reconstruire les affectées ─────┤   entreprise + entreprise_silver
        │                                      │
        └─ 4. journaliser l'extrait ──────────┘   kbo_update_log
```

Pourquoi ? Un `establishment_delete.csv` ne contient que `EstablishmentNumber`.
Pour identifier **quelle entreprise** doit voir sa fiche rafraîchie, il faut
lire `kbo_establishment` — ce qui devient impossible une fois la ligne supprimée.
Inverser 1 et 2, c'est laisser des documents `entreprise` contenant des
établissements qui n'existent plus. Le pipeline ne lèverait aucune erreur :
il produirait simplement des données fausses.

---

## Configuration

La cellule ci-dessous porte le tag `parameters` : c'est le point d'injection de
[papermill](https://papermill.readthedocs.io/), utilisé par le DAG Airflow en
fin de notebook. `UPDATE_DIR = None` signifie « traiter tous les extraits en
attente, dans l'ordre ».

In [ ]:
UPDATE_DIR = None        # None = tous les extraits en attente ; sinon un chemin precis
UPDATES_ROOT = None      # dossier contenant les KboOpenData_*_Update (defaut : ce dossier)

In [ ]:
import csv
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pymongo
from pymongo import DeleteMany, InsertOne, ReplaceOne

sys.path.insert(0, str(Path.cwd()))
from kbo_lib import SilverTransformer, bronze_stages

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27018")
MONGO_DB = os.getenv("MONGO_DB", "kbo")

db = pymongo.MongoClient(MONGO_URI, socketTimeoutMS=None)[MONGO_DB]
ROOT = Path(UPDATES_ROOT) if UPDATES_ROOT else Path.cwd()

print("mongodb :", MONGO_URI, "->", MONGO_DB)
print("extraits:", ROOT)
for name in ("entreprise", "entreprise_silver", "kbo_enterprise"):
    print(f"  {name:<20} {db[name].estimated_document_count():>12,}")
print("\nextrait de base :",
      {d["Variable"]: d["Value"] for d in db.kbo_meta.find({}, {"_id": 0})
       if d["Variable"] in ("ExtractNumber", "ExtractType", "SnapshotDate")})

### Index nécessaires

Deux index font défaut pour les mises à jour : les collections `kbo_establishment`
et `kbo_branch` sont indexées sur `EnterpriseNumber` (pour les jointures), mais
pas sur **leur propre clé**. C'est pourtant exactement ce dont on a besoin ici :
retrouver un établissement par son numéro avant de le supprimer.

C'est typique d'un passage en production : les index suffisants pour un chargement
initial ne sont pas forcément ceux qu'exige une mise à jour incrémentale.

In [ ]:
for collection, field in [("kbo_establishment", "EstablishmentNumber"),
                          ("kbo_branch", "Id"),
                          ("kbo_update_log", "appliedAt")]:
    print(f"  {collection:<20} {db[collection].create_index(field)}")

---

## 1. Lecture de `meta.csv` et des fichiers insert/delete

Une classe dédiée encapsule un dossier d'extrait : elle lit `meta.csv`, expose
`inserts(entite)` / `deletes(entite)`, et rien de plus. Les fichiers sont lus
à la demande — inutile de charger 10 000 lignes d'activités pour consulter le
numéro d'extrait.

Le tableau `ENTITIES` constitue la seule description du modèle : nom de fichier,
collection cible, clé. Tout le reste du notebook en dérive, ce qui évite les
sept blocs copiés-collés que l'on rencontre habituellement dans ce type de script.

In [ ]:
# entite -> (collection bronze, cle de ligne, regime)
ENTITIES = {
    "enterprise":    ("kbo_enterprise",    "EnterpriseNumber",    "keyed"),
    "establishment": ("kbo_establishment", "EstablishmentNumber", "keyed"),
    "branch":        ("kbo_branch",        "Id",                  "keyed"),
    "denomination":  ("kbo_denomination",  "EntityNumber",        "whole"),
    "address":       ("kbo_address",       "EntityNumber",        "whole"),
    "contact":       ("kbo_contact",       "EntityNumber",        "whole"),
    "activity":      ("kbo_activity",      "EntityNumber",        "whole"),
}


def read_csv(path: Path) -> list[dict]:
    """Lit un CSV de l'extrait ; un fichier absent vaut liste vide."""
    if not path.exists():
        return []
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


class Extract:
    """Un dossier `KboOpenData_XXXX_..._Update`."""

    def __init__(self, folder: Path):
        self.folder = Path(folder)
        rows = read_csv(self.folder / "meta.csv")
        self.meta = {row["Variable"]: row["Value"] for row in rows}
        self.number = int(self.meta["ExtractNumber"])
        self.type = self.meta["ExtractType"]
        self.snapshot = self.meta["SnapshotDate"]

    def inserts(self, entity: str) -> list[dict]:
        return read_csv(self.folder / f"{entity}_insert.csv")

    def deletes(self, entity: str) -> list[dict]:
        return read_csv(self.folder / f"{entity}_delete.csv")

    def summary(self) -> dict:
        return {entity: (len(self.inserts(entity)), len(self.deletes(entity)))
                for entity in ENTITIES}

    def __repr__(self) -> str:
        return f"<Extract {self.number} {self.type} snapshot={self.snapshot}>"


def discover(root: Path) -> list[Extract]:
    """Tous les extraits presents, tries par numero croissant."""
    folders = [p for p in root.iterdir() if p.is_dir() and (p / "meta.csv").exists()]
    return sorted((Extract(p) for p in folders), key=lambda e: e.number)


extracts = discover(ROOT)
print(f"{len(extracts)} extrait(s) trouve(s) :\n")
for extract in extracts:
    print(f"  {extract}")

In [ ]:
print(f"{'entite':<16}{'inserts':>10}{'deletes':>10}   regime")
for extract in extracts:
    print(f"\n--- extrait {extract.number} (snapshot {extract.snapshot}) ---")
    for entity, (inserted, deleted) in extract.summary().items():
        regime = ENTITIES[entity][2]
        print(f"{entity:<16}{inserted:>10,}{deleted:>10,}   "
              f"{'par cle' if regime == 'keyed' else 'entite entiere'}")

---

## 2bis. L'ensemble affecté — calculé en premier

Cette étape intervient **avant** l'application (section 2), pour la raison
expliquée ci-dessus. Trois sources d'entreprises affectées :

1. **`enterprise_insert` / `enterprise_delete`** → le numéro est directement disponible.
2. **`establishment` / `branch`** → l'insert porte `EnterpriseNumber` ; le
   delete ne porte que la clé de l'entité fille, il faut donc **résoudre le
   propriétaire dans le bronze avant suppression**.
3. **Les 4 tables de détail** → elles portent un `EntityNumber` qui peut
   désigner les trois niveaux. On le discrimine par son préfixe :

| Préfixe | Niveau | Résolution |
|---|---|---|
| `2.` | établissement | `kbo_establishment` → `EnterpriseNumber` |
| `9.` | succursale | `kbo_branch` → `EnterpriseNumber` |
| sinon | entreprise | c'est déjà le bon numéro |

> Une entreprise **supprimée** figure également dans l'ensemble affecté : il faut
> la retirer de `entreprise` et `entreprise_silver`, ce qui constitue un
> traitement à part entière.

In [ ]:
def resolve_owners(entity_numbers: set[str]) -> set[str]:
    """Numeros d'entite (3 niveaux melanges) -> numeros d'entreprise proprietaires."""
    owners, establishments, branches = set(), [], []
    for number in entity_numbers:
        if number.startswith("2."):
            establishments.append(number)
        elif number.startswith("9."):
            branches.append(number)
        else:
            owners.add(number)          # c'est deja une entreprise

    if establishments:
        owners |= {d["EnterpriseNumber"] for d in db.kbo_establishment.find(
            {"EstablishmentNumber": {"$in": establishments}}, {"EnterpriseNumber": 1})}
    if branches:
        owners |= {d["EnterpriseNumber"] for d in db.kbo_branch.find(
            {"Id": {"$in": branches}}, {"EnterpriseNumber": 1})}
    return owners


def affected_enterprises(extract: Extract) -> set[str]:
    """Entreprises dont la fiche devra etre reconstruite.

    A APPELER AVANT d'appliquer les suppressions : les deletes des entites
    filles ne portent que leur propre cle, le lien vers l'entreprise ne se lit
    que dans le bronze encore intact.
    """
    affected = set()

    # 1. entreprises citees directement
    affected |= {r["EnterpriseNumber"] for r in extract.inserts("enterprise")}
    affected |= {r["EnterpriseNumber"] for r in extract.deletes("enterprise")}

    # 2. entites filles : insert -> colonne EnterpriseNumber, delete -> lookup
    for entity, key, collection in (
            ("establishment", "EstablishmentNumber", "kbo_establishment"),
            ("branch", "Id", "kbo_branch")):
        affected |= {r["EnterpriseNumber"] for r in extract.inserts(entity)}
        keys = [r[key] for r in extract.deletes(entity)]
        if keys:
            affected |= {d["EnterpriseNumber"] for d in db[collection].find(
                {key: {"$in": keys}}, {"EnterpriseNumber": 1})}

    # 3. tables de detail : EntityNumber melange les trois niveaux
    entity_numbers = set()
    for entity, (_, _, regime) in ENTITIES.items():
        if regime != "whole":
            continue
        entity_numbers |= {r["EntityNumber"] for r in extract.inserts(entity)}
        entity_numbers |= {r["EntityNumber"] for r in extract.deletes(entity)}
    affected |= resolve_owners(entity_numbers)

    return affected

In [ ]:
demo = extracts[0]
affected = affected_enterprises(demo)
print(f"extrait {demo.number} : {len(affected):,} entreprises affectees")
print(f"soit {len(affected) / db.kbo_enterprise.estimated_document_count():.4%} de la base")
print(f"\nechantillon : {sorted(affected)[:6]}")

orphans = len(affected) - db.kbo_enterprise.count_documents({"_id": {"$in": list(affected)}})
print(f"\ndont inconnues du bronze (creations a venir) : {orphans}")

---

## 2. Appliquer la mise à jour sur bronze

Pour chacune des 7 entités, appliquez les opérations insert/delete sur sa
collection brute (`kbo_enterprise`, `kbo_establishment`, `kbo_branch`,
`kbo_denomination`, `kbo_address`, `kbo_contact`, `kbo_activity`) :

### Le détail qui garantit la sécurité du rejeu

Pour le régime « entité entière », on purge les `EntityNumber` du fichier
delete **et** ceux du fichier insert :

```python
purge = deleted_entities | inserted_entities
```

Cette union n'est pas une précaution superflue. Sans elle, rejouer un extrait
insérerait une seconde copie de chaque ligne d'une entité présente dans `insert`
mais absente de `delete`. Avec elle, l'opération est **idempotente** par
construction — exactement la propriété que la section 4 exige.

Pour le régime « par clé », `ReplaceOne(..., upsert=True)` offre la même
garantie : rejouer remplace le document par lui-même.

Tout passe par un seul `bulk_write` par collection : un seul aller-retour réseau
au lieu de plusieurs milliers.

In [ ]:
def apply_to_bronze(extract: Extract, *, verbose: bool = True) -> dict:
    """Applique deletes puis inserts sur les 7 collections brutes."""
    report = {}

    for entity, (collection_name, key, regime) in ENTITIES.items():
        inserts = extract.inserts(entity)
        deletes = extract.deletes(entity)
        if not inserts and not deletes:
            report[entity] = {"deleted": 0, "inserted": 0}
            continue

        operations = []
        if regime == "keyed":
            keys = [row[key] for row in deletes]
            if keys:
                operations.append(DeleteMany({key: {"$in": keys}}))
            for row in inserts:
                # `kbo_enterprise` utilise le numero comme _id (cle naturelle)
                document = dict(row)
                if collection_name == "kbo_enterprise":
                    document["_id"] = row[key]
                operations.append(ReplaceOne({key: row[key]}, document, upsert=True))
        else:
            # purge de l'entite entiere : delete + insert, pour l'idempotence
            purge = ({row["EntityNumber"] for row in deletes}
                     | {row["EntityNumber"] for row in inserts})
            if purge:
                operations.append(DeleteMany({"EntityNumber": {"$in": list(purge)}}))
            operations.extend(InsertOne(dict(row)) for row in inserts)

        result = db[collection_name].bulk_write(operations, ordered=True)
        report[entity] = {"deleted": result.deleted_count,
                          "inserted": result.inserted_count + result.upserted_count,
                          "replaced": result.modified_count}
        if verbose:
            print(f"  {entity:<16} -{result.deleted_count:>6,}  "
                  f"+{result.inserted_count + result.upserted_count:>6,}  "
                  f"~{result.modified_count:>6,}")
    return report

---

## 3. Propager vers `entreprise` et `entreprise_silver` — UNIQUEMENT pour les entreprises affectées

Un rebuild complet de `entreprise_silver` (suppression + réinsertion de toute
la base) serait inutilement coûteux pour un lot qui ne touche que quelques
centaines d'entreprises sur des millions.

Attention à l'ORDRE : calculez l'ensemble affecté AVANT d'avoir appliqué les
deletes de l'étape précédente — sans quoi il ne sera plus possible de retrouver
le propriétaire d'un établissement ou d'une succursale déjà supprimé.

### `$merge` et non `$out`

C'est **la** différence entre le build complet et l'incrémental. `$out`
**remplace la collection entière** : l'utiliser ici détruirait les 1,95 million
d'entreprises pour n'en réécrire que quelques centaines.

`$merge` écrit document par document :

```python
{"$merge": {"into": "entreprise", "on": "_id",
            "whenMatched": "replace",      # et surtout pas "merge"
            "whenNotMatched": "insert"}}
```

`whenMatched: "replace"` est tout aussi important. Avec `"merge"`, les tableaux
du document existant seraient conservés et fusionnés — une activité supprimée
resterait indéfiniment dans la fiche. On veut le document **entièrement
reconstruit**, pas simplement complété.

### Le pipeline « lent » redevient le bon choix

Le notebook bronze avait mesuré que le pipeline à `$lookup` imbriqués tournait
à ~69 doc/s, soit 8 h sur toute la base — d'où la construction en trois passes
avec collections intermédiaires.

En incrémental, le calcul est radicalement différent : **quelques centaines
d'entreprises à 69 doc/s, c'est quelques secondes**. On réutilise donc
directement `bronze_stages()` du module partagé, sans maintenir aucune
collection intermédiaire.

Une optimisation n'est jamais bonne dans l'absolu : elle est bonne pour un
volume donné.

### Trois cas à gérer

| Cas | Traitement |
|---|---|
| entreprise modifiée | reconstruite dans `entreprise` puis `entreprise_silver` |
| entreprise créée | idem — `$merge` l'insère |
| entreprise **supprimée** | retirée des deux collections |

In [ ]:
transformer = SilverTransformer.from_db(db)
print(f"{len(transformer.codes):,} codes de traduction charges")


def refresh_enterprises(numbers: set[str], *, verbose: bool = True) -> dict:
    """Reconstruit `entreprise` puis `entreprise_silver` pour ces entreprises."""
    if not numbers:
        return {"rebuilt": 0, "removed": 0}

    target = list(numbers)
    alive = {d["_id"] for d in db.kbo_enterprise.find({"_id": {"$in": target}}, {"_id": 1})}
    removed = numbers - alive

    # 1. entreprises disparues du bronze : on les retire des deux couches
    if removed:
        db.entreprise.delete_many({"_id": {"$in": list(removed)}})
        db.entreprise_silver.delete_many({"_id": {"$in": list(removed)}})

    # 2. les autres : rejointure ciblee, puis $merge (jamais $out)
    if alive:
        db.kbo_enterprise.aggregate([
            {"$match": {"_id": {"$in": list(alive)}}},
            *bronze_stages(),
            {"$merge": {"into": "entreprise", "on": "_id",
                        "whenMatched": "replace", "whenNotMatched": "insert"}},
        ], allowDiskUse=True)

        # 3. silver : memes regles que le build complet (module partage)
        operations = [
            ReplaceOne({"_id": document["_id"]}, document, upsert=True)
            for document in (transformer.to_silver(bronze)
                             for bronze in db.entreprise.find({"_id": {"$in": list(alive)}}))
        ]
        if operations:
            db.entreprise_silver.bulk_write(operations, ordered=False)

    if verbose:
        print(f"  reconstruites : {len(alive):>6,}   supprimees : {len(removed):>6,}")
    return {"rebuilt": len(alive), "removed": len(removed)}

---

## 4. Ne pas rejouer deux fois le même extrait

Créez une collection `kbo_update_log` qui enregistre, pour chaque extrait
appliqué avec succès : `extractNumber`, `extractType`, `snapshotDate`, et
l'horodatage de l'application.

### Deux garde-fous, pas un

L'énoncé demande d'éviter le **rejeu**. Les extraits KBO étant des deltas, il
faut en réalité se prémunir de deux erreurs symétriques :

| Risque | Garde-fou |
|---|---|
| appliquer deux fois le même extrait | `_id = extractNumber` → déjà présent = on saute |
| appliquer 434 sans avoir appliqué 433 | on exige `numéro == dernier appliqué + 1` |

Le second est le plus dangereux : sauter un extrait ne provoque **aucune
erreur**, il laisse simplement la base dans un état silencieusement faux. Un
delta n'est valide que posé sur l'état exact qu'il attend.

Le point de départ de la chaîne est l'extrait complet **431**, lu dans
`kbo_meta` — la couche bronze connaît donc d'elle-même sa position dans la
séquence.

> **Sur l'atomicité.** MongoDB ne permet pas d'englober un `$merge` sur
> plusieurs millions de documents dans une transaction. Le journal est donc
> écrit **après** le succès complet : en cas d'interruption en cours de route,
> l'extrait est rejoué au prochain lancement. C'est possible précisément parce
> que chaque opération est idempotente (section 2). On opte pour *at-least-once*
> plutôt qu'*exactly-once*, ce qui est le compromis habituel lorsque la
> transaction distribuée n'est pas disponible.

In [ ]:
def last_applied() -> int:
    """Numero du dernier extrait applique, ou celui du full de depart."""
    latest = db.kbo_update_log.find_one(sort=[("_id", -1)])
    if latest:
        return latest["_id"]
    base = db.kbo_meta.find_one({"Variable": "ExtractNumber"})
    return int(base["Value"]) if base else 0


def already_applied(extract: Extract) -> bool:
    return db.kbo_update_log.find_one({"_id": extract.number}) is not None


def log_extract(extract: Extract, report: dict) -> None:
    db.kbo_update_log.replace_one(
        {"_id": extract.number},
        {"_id": extract.number,
         "extractNumber": extract.number,
         "extractType": extract.type,
         "snapshotDate": extract.snapshot,
         "appliedAt": datetime.now(timezone.utc),
         **report},
        upsert=True)


print("dernier extrait applique :", last_applied())

### L'orchestrateur

`apply_extract` enchaîne les quatre étapes dans l'ordre imposé, et refuse
d'exécuter un extrait si les garde-fous ne sont pas satisfaits.

In [ ]:
def apply_extract(extract: Extract, *, verbose: bool = True) -> dict:
    """Applique un extrait de bout en bout, ou explique pourquoi il est refuse."""
    if already_applied(extract):
        print(f"[{extract.number}] deja applique - ignore")
        return {"status": "skipped"}

    expected = last_applied() + 1
    if extract.number != expected:
        print(f"[{extract.number}] REFUSE : l'extrait attendu est {expected}. "
              f"Appliquer un delta hors sequence corromprait la base.")
        return {"status": "out-of-sequence", "expected": expected}

    started = time.perf_counter()
    print(f"[{extract.number}] snapshot {extract.snapshot}")

    # 1. AVANT toute suppression
    affected = affected_enterprises(extract)
    print(f"  {len(affected):,} entreprises affectees")

    # 2. bronze brut
    bronze_report = apply_to_bronze(extract, verbose=verbose)

    # 3. propagation ciblee
    refresh_report = refresh_enterprises(affected, verbose=verbose)

    # 4. journal (apres succes complet)
    report = {"affected": len(affected),
              "rebuilt": refresh_report["rebuilt"],
              "removed": refresh_report["removed"],
              "durationSeconds": round(time.perf_counter() - started, 1),
              "bronze": bronze_report}
    log_extract(extract, report)

    print(f"  applique en {report['durationSeconds']}s")
    return {"status": "applied", **report}

---

## 5. Application des trois extraits, dans l'ordre

`discover()` trie déjà par numéro croissant : 432, puis 433, puis 434. Le
garde-fou de séquence vérifie ce tri plutôt que de lui faire confiance
aveuglément.

In [ ]:
selected = [Extract(Path(UPDATE_DIR))] if UPDATE_DIR else extracts

results = []
for extract in selected:
    results.append(apply_extract(extract))
    print()

print("=" * 58)
for extract, result in zip(selected, results):
    print(f"  extrait {extract.number} : {result['status']:<16} "
          f"{result.get('affected', ''):>7} affectees  "
          f"{result.get('durationSeconds', '')!s:>6}s")

### Historique des applications

In [ ]:
print(f"{'extrait':>8} {'snapshot':>12} {'affectees':>10} {'reconstr.':>10} "
      f"{'suppr.':>8} {'duree':>7}  applique le")
for entry in db.kbo_update_log.find().sort("_id", 1):
    print(f"{entry['_id']:>8} {entry['snapshotDate']:>12} {entry['affected']:>10,} "
          f"{entry['rebuilt']:>10,} {entry['removed']:>8,} "
          f"{entry['durationSeconds']:>6}s  "
          f"{entry['appliedAt']:%Y-%m-%d %H:%M:%S}")

### Test d'idempotence

On relance l'ensemble du traitement. Chaque extrait doit être refusé par le
journal, sans qu'une seule écriture ne soit envoyée vers la base.

In [ ]:
before = {name: db[name].estimated_document_count()
          for name in ("kbo_enterprise", "kbo_activity", "entreprise", "entreprise_silver")}

for extract in selected:
    apply_extract(extract)

after = {name: db[name].estimated_document_count() for name in before}
print(f"\n{'collection':<22}{'avant':>14}{'apres':>14}   ecart")
for name in before:
    print(f"{name:<22}{before[name]:>14,}{after[name]:>14,}   {after[name] - before[name]:>+,}")

---

## 6. Vérification

Trois contrôles : la cohérence entre les couches, la traçabilité d'une
modification réelle de bout en bout, et l'absence de documents fantômes.

In [ ]:
counts = {name: db[name].count_documents({}) for name in
          ("kbo_enterprise", "entreprise", "entreprise_silver")}
print("volumetrie des trois couches :")
for name, count in counts.items():
    print(f"  {name:<20} {count:>12,}")
aligned = len(set(counts.values())) == 1
print(f"  -> {'ALIGNEES' if aligned else 'ECART DETECTE'}")

# Recherche de fantomes sur le PERIMETRE TOUCHE : une entreprise supprimee du
# brut mais restee dans `entreprise` ou `entreprise_silver`. On se limite aux
# entreprises affectees -- un $nin sur 1,95 M de valeurs serait absurde.
touched = set()
for extract in selected:
    touched |= {r["EnterpriseNumber"] for r in extract.deletes("enterprise")}

if touched:
    alive = {d["_id"] for d in db.kbo_enterprise.find({"_id": {"$in": list(touched)}}, {"_id": 1})}
    deleted = touched - alive
    ghosts_bronze = db.entreprise.count_documents({"_id": {"$in": list(deleted)}})
    ghosts_silver = db.entreprise_silver.count_documents({"_id": {"$in": list(deleted)}})
    print(f"\n{len(deleted):,} entreprises supprimees par les extraits")
    print(f"  restees dans `entreprise`        : {ghosts_bronze}   (attendu 0)")
    print(f"  restees dans `entreprise_silver` : {ghosts_silver}   (attendu 0)")

In [ ]:
# Une entreprise reellement modifiee par le dernier extrait : on la suit
# depuis le CSV jusqu'au document silver.
last_extract = selected[-1]
sample_row = last_extract.inserts("enterprise")[0]
number = sample_row["EnterpriseNumber"]

print(f"entreprise {number}\n")
print("1. ligne du CSV d'insert :")
print("  ", sample_row)

print("\n2. document brut (kbo_enterprise) :")
print("  ", db.kbo_enterprise.find_one({"_id": number}))

silver = db.entreprise_silver.find_one({"_id": number})
print("\n3. document silver (traduit) :")
for key in ("status", "juridicalSituation", "juridicalForm", "typeOfEnterprise"):
    print(f"   {key:<20} {silver.get(key)}")
print(f"   denominations        {list(silver.get('denominations', {}))}")
print(f"   etablissements       {len(silver.get('establishments', {}))}")

In [ ]:
# Le JuridicalSituation du CSV doit se retrouver traduit dans le silver.
expected = transformer.translate("JuridicalSituation", sample_row["JuridicalSituation"])
actual = silver.get("juridicalSituation")
print(f"CSV JuridicalSituation = {sample_row['JuridicalSituation']!r}")
print(f"  attendu apres traduction : {expected!r}")
print(f"  present dans le silver   : {actual!r}")
print(f"  -> {'COHERENT' if expected == actual else 'INCOHERENT'}")

---

## 7. Bonus — orchestration avec Airflow

Le notebook applique correctement les extraits, mais il faut encore que quelqu'un
le déclenche chaque jour, dans le bon ordre, et sache quoi faire en cas d'échec.
C'est le rôle d'un orchestrateur.

### Ce qu'Airflow apporte réellement ici

| Besoin | Réponse Airflow |
|---|---|
| lancer tous les jours | `schedule="0 6 * * *"` |
| ne pas superposer deux exécutions | `max_active_runs=1` |
| réessayer après une panne réseau | `retries` + `retry_delay` |
| historique et logs par exécution | interface web |
| reprendre après incident | le journal `kbo_update_log` fait foi |

### Ce qu'il n'apporte pas, et qu'il ne faut pas lui déléguer

**La correction ne vient pas de l'orchestrateur.** Les garde-fous — ordre de
séquence, protection contre le rejeu, idempotence — sont dans le notebook. Un
DAG qui compterait sur Airflow pour ne pas rejouer un extrait serait incorrect
dès la première reprise manuelle. Airflow *déclenche* ; il ne garantit rien sur
la donnée.

C'est pourquoi le DAG est volontairement minimaliste : il ne réimplémente
aucune logique métier.

### Le DAG

Le fichier est livré dans `airflow/dags/kbo_update_dag.py`.

```
list_pending  ──>  apply_updates  ──>  report
```

- **`list_pending`** compare les dossiers présents au contenu de
  `kbo_update_log`, et retourne la liste triée des extraits en attente.
- **`apply_updates`** exécute le notebook via **papermill**, un extrait à la
  fois, **séquentiellement**. Le notebook exécuté est archivé : chaque
  exécution laisse une trace auditable de ce qui a réellement tourné.
- **`report`** relit le journal et publie un résumé.

> **Pourquoi papermill plutôt que réécrire la logique dans le DAG ?** Pour
> éviter deux implémentations à maintenir. Le notebook est la source de vérité,
> le DAG l'exécute avec des paramètres. Toute correction de règle métier
> s'effectue à un seul endroit.
>
> L'exécution **séquentielle** est imposée : le *dynamic task mapping*
> d'Airflow paralléliserait les extraits, ce qui violerait la contrainte d'ordre.

In [ ]:
dag_path = Path("airflow/dags/kbo_update_dag.py")
print(dag_path.read_text(encoding="utf-8") if dag_path.exists()
      else f"(DAG livre dans {dag_path})")

---

## Bilan

Trois extraits journaliers appliqués sur une base de 1,95 million d'entreprises,
sans reconstruire quoi que ce soit d'inutile.

### Les décisions qui comptent

| Décision | Raison |
|---|---|
| Ensemble affecté calculé **avant** les deletes | après suppression, le lien fille → entreprise est perdu |
| Purge sur `delete ∪ insert` | rend l'opération idempotente, sécurisant le rejeu |
| `$merge` + `whenMatched: replace` | `$out` détruirait la collection ; `merge` laisserait des données périmées |
| Pipeline à `$lookup` imbriqués réutilisé | lent sur 1,95 M, quasi instantané sur quelques centaines |
| Règles silver dans `kbo_lib.py` | une seule définition pour le build complet et l'incrémental |
| Journal écrit **après** succès | *at-least-once* assumé, viable car tout est idempotent |
| Contrôle de séquence `n = dernier + 1` | un extrait sauté ne lève aucune erreur, il fausse silencieusement la base |
| DAG minimaliste | la correction appartient au pipeline, pas à l'orchestrateur |

### Ce qu'il resterait à faire pour de la production

- **Récupération automatique** des extraits depuis le portail KBO (aujourd'hui déposés manuellement).
- **Alerte sur extrait manquant** : si le numéro attendu n'est pas disponible depuis 48 h, mieux vaut être prévenu plutôt que de l'ignorer.
- **Vérification de bout en bout** : recharger périodiquement un extrait *full* et le comparer, afin de détecter une dérive accumulée par les deltas.